# Examen Final - Caso de Estudio 2: Segmentación de Clientes y Reglas de Asociación
Este notebook simula un caso práctico de examen final centrado en el **Aprendizaje No Supervisado**:
1. **Segmentación de Clientes (K-Means):**
   - Preprocesamiento de variables y escalado con `StandardScaler`.
   - Reducción de dimensiones con **PCA** (Análisis de Componentes Principales).
   - Optimización del número de clusters $K$: **Método del Codo (Inercia/WCSS)** y **Coeficiente de Silueta**.
   - Caracterización e interpretación comercial de los clusters resultantes.
2. **Reglas de Asociación (Algoritmo Apriori):**
   - Preparación de datos transaccionales (simulando compras de la tienda).
   - Aplicación de las métricas de **Soporte, Confianza y Lift** para extraer reglas valiosas.
3. **Banco de Preguntas Teórico-Prácticas de Examen** al final del notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Para reglas de asociación (Instalar si no se tiene: %pip install mlxtend)
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

import warnings
warnings.filterwarnings('ignore')


## 1. Segmentación de Clientes (K-Means Clustering)


In [ ]:
# Carga de datos de clientes de la tienda
# Nota: Asegúrate de tener 'clientes_tienda.csv' en la carpeta Sem_12
df = pd.read_csv('Sem_12/clientes_tienda.csv')
print("Dimensiones:", df.shape)
df.head()


In [ ]:
# Seleccionamos las columnas numéricas clave para segmentar
# Típicamente: Age (Edad), Annual Income (Ingreso Anual en miles) y Spending Score (Puntuación de Gasto de 1-100)
# Ajusta los nombres de columna según corresponda en tu dataset.
# Asumiremos columnas típicas: 'Age', 'Annual Income (k$)', 'Spending Score (1-100)'
col_segmentacion = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X = df[col_segmentacion]

# Escalado obligatorio
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Reducción de dimensionalidad con PCA para visualización posterior
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada acumulada por los 2 componentes del PCA:", sum(pca.explained_variance_ratio_)*100, "%")


### Determinación de K óptimo


In [ ]:
# A. Método del Codo (Inercia/WCSS)
wcss = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, wcss, 'bo-', linewidth=2)
plt.title('Método del Codo (Elbow Method)')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia (WCSS)')
plt.grid(True)
plt.show()


In [ ]:
# B. Método de Coeficiente de Silueta
sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    print(f"K = {k} | Coeficiente de Silueta = {score:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(k_range, sil_scores, 'ro-', linewidth=2)
plt.title('Análisis de Silueta por K')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Coeficiente de Silueta Promedio')
plt.grid(True)
plt.show()


### Entrenamiento y Visualización de K-Means final (elegimos K=5)


In [ ]:
# Asumiendo que K=5 es el óptimo
k_optimo = 5
kmeans_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df['Cluster'] = kmeans_final.fit_predict(X_scaled)

# Graficamos los clusters en el espacio 2D del PCA
plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['Cluster'], palette='Set1', s=100)
plt.title('Segmentación de Clientes visualizada en 2D (PCA)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Cluster')
plt.show()


In [ ]:
# Caracterización promedio de cada Cluster
df.groupby('Cluster')[col_segmentacion].mean()


## 2. Reglas de Asociación (Algoritmo Apriori)


In [ ]:
# Simulación de un dataset de transacciones para la tienda (Compras cruzadas)
dataset = [
    ['Pan', 'Leche', 'Pañales', 'Cerveza', 'Huevos'],
    ['Leche', 'Pañales', 'Cerveza', 'Cola'],
    ['Pan', 'Leche', 'Pañales', 'Cerveza'],
    ['Pan', 'Leche', 'Pañales', 'Huevos'],
    ['Pan', 'Cola', 'Pañales', 'Cerveza'],
    ['Leche', 'Pañales', 'Cerveza', 'Huevos', 'Cola'],
    ['Pan', 'Leche', 'Huevos'],
    ['Pan', 'Leche', 'Pañales', 'Cerveza', 'Huevos']
]

# One-hot encoding de las transacciones para usar mlxtend
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df_trans = pd.DataFrame(te_ary, columns=te.columns_)
df_trans.head()


In [ ]:
# Encontrar conjuntos de ítems frecuentes (Soporte Mínimo = 0.3)
frequent_itemsets = apriori(df_trans, min_support=0.3, use_colnames=True)
frequent_itemsets.sort_values(by='support', ascending=False)


In [ ]:
# Generar Reglas de Asociación usando Confianza Mínima = 0.7
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)
# Filtramos y ordenamos por Lift para encontrar las reglas más fuertes
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values(by='lift', ascending=False)


## 3. Banco de Preguntas y Respuestas Teórico-Prácticas (Nivel Examen)

### **Pregunta 1 (Métodos de Agrupamiento):**
**Enunciado:** Si el Método del Codo y el Coeficiente de Silueta sugieren números de clusters distintos (ej: el codo insinúa K=3 pero la Silueta máxima se da en K=5), ¿cómo resolvería esta contradicción para tomar una decisión en un ámbito empresarial real?

**Respuesta Correcta y Justificación:**
Ambas métricas miden aspectos diferentes:
- La **Inercia (WCSS)** busca minimizar la distancia intra-cluster (cohesión pura), pero siempre disminuye a medida que aumentas K.
- El **Coeficiente de Silueta** mide la cohesión y la separación simultáneamente.
En un ámbito empresarial, se prefiere la solución que ofrezca **mayor interpretabilidad comercial**. 
- Si K=5 separa a los clientes en grupos muy claros con perfiles de marketing específicos (ej: jóvenes gastadores, adultos conservadores, etc.) y la silueta es alta, se elige K=5.
- Si K=3 agrupa perfiles demasiado heterogéneos y difíciles de perfilar comercialmente, no es útil aunque el codo se marque ahí. La silueta suele ser un mejor indicador matemático de separación real.

---

### **Pregunta 2 (Efecto del Escalado en K-Means):**
**Enunciado:** Describa paso a paso qué le ocurriría a su modelo de segmentación de clientes de la tienda si decide no aplicar `StandardScaler` en las variables `Age` (rango 18-70), `Annual Income` (rango 15-137 en miles) y `Spending Score` (rango 1-100).

**Respuesta Correcta y Justificación:**
K-Means utiliza la distancia euclidiana:
\[d(p, q) = \sqrt{\sum (p_i - q_i)^2}\]
Al no escalar:
- La variable `Annual Income` (en miles de dólares, diferencia de hasta 120 unidades) y la variable `Spending Score` (diferencia de hasta 100 unidades) tendrán un impacto geométrico enorme en la distancia en comparación con `Age` (diferencia máxima de 52 unidades).
- Los centroides se moverán casi exclusivamente en el eje de los ingresos. El algoritmo agrupará a los clientes según su riqueza ignorando su edad, perdiendo una dimensión valiosa para la caracterización.

---

### **Pregunta 3 (Interpretación del Lift):**
**Enunciado:** En la tabla de reglas de asociación obtenida, se observa una regla con un Lift de **1.45**. Explique qué significa este valor numérico en el contexto de las compras de los clientes y qué estrategia de colocación de productos recomendaría al gerente del supermercado.

**Respuesta Correcta y Justificación:**
Un **Lift de 1.45** significa que la probabilidad de que un cliente compre el producto consecuente aumenta en un **45%** si ya ha colocado el producto antecedente en su carrito, en comparación con la probabilidad de comprar el consecuente de forma aislada (al azar). Esto demuestra una fuerte asociación positiva.
*   **Estrategia Comercial:** 
    *   *Venta Cruzada:* Colocar los productos físicamente distantes en la tienda para obligar al cliente a recorrer el pasillo y tentar compras impulsivas intermedias, pero promocionarlos juntos en cupones de descuento.
    *   *Empaquetado (Bundling):* Crear un paquete promocional que incluya ambos productos a un costo ligeramente inferior al de la suma individual.

---

### **Pregunta 4 (Pregunta Hipotética sobre PCA):**
**Enunciado:** Para graficar los clusters de K-Means, usted aplicó PCA para reducir las variables a 2 dimensiones. ¿Se entrenó el algoritmo K-Means sobre las variables originales escaladas o sobre los componentes principales del PCA? Explique los pros y contras de cada enfoque de cara a la interpretación de los perfiles.

**Respuesta Correcta y Justificación:**
En este notebook, **K-Means se entrenó sobre las variables originales escaladas** (`X_scaled`), y el PCA se utilizó exclusivamente como una herramienta de visualización para graficar en 2D.
*   **Entrenar en variables originales (Usado aquí):**
    *   *Pro:* Facilidad de interpretación. Al caracterizar los clusters con `.mean()`, obtenemos la edad y el ingreso promedio directo de los clientes reales de ese grupo.
    *   *Contra:* Si hay demasiadas variables correlacionadas, puede introducir ruido geométrico al clustering.
*   **Entrenar en componentes de PCA:**
    *   *Pro:* Reduce el ruido y elimina la multicolinealidad, lo que hace que K-Means corra más rápido en datasets gigantescos.
    *   *Contra:* Se pierde la interpretabilidad directa. Los centroides quedan definidos en términos de "Componente 1" y "Componente 2", los cuales son combinaciones matemáticas complejas lineales de las variables originales y difíciles de explicar al negocio.

---

### **Pregunta 5 (Limitaciones de Reglas de Asociación):**
**Enunciado:** Si el algoritmo Apriori genera 500 reglas de asociación basadas en su soporte y confianza mínimos, ¿por qué no es recomendable implementarlas todas a la vez en la tienda y qué métrica adicional usaría para filtrar las reglas más valiosas?

**Respuesta Correcta y Justificación:**
Implementar 500 reglas a la vez saturará visualmente al cliente (demasiadas ofertas) y complicará la logística de la tienda. Además, muchas de estas reglas pueden ser "triviales" (ej: *Pan de molde $ightarrow$ Mantequilla*, que se comprarían juntos de todos modos sin necesidad de promoción).
*   **Filtro:** Se debe filtrar utilizando el **Lift** y el **Soporte**. Se priorizan las reglas con **mayor Lift** (que representan las asociaciones menos obvias y con mayor impacto de incentivo de compra) y se exige un **Soporte mínimo razonable** para asegurar que la regla aplique a un porcentaje significativo de transacciones del negocio, justificando el costo de la campaña comercial.
